# Plot Enso Synthetic Mvo V4P

Model-versus-observation synthetic metrics: v4P comparison with CMIP.


In [ ]:
from pathlib import Path
import importlib
import sys

PROJECT_ROOT = next(
    (path for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
     if (path / "scripts").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not find the repository scripts directory.")

SCRIPTS_DIR = PROJECT_ROOT / "scripts"
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

import metrics_group_merger
import synthetic_metrics_workflow
importlib.reload(metrics_group_merger)
importlib.reload(synthetic_metrics_workflow)

from metrics_group_merger import MetricsGroup, PCMDIRun, merge_metrics_group
from synthetic_metrics_workflow import (
    dataset_for_merged_group,
    make_comparison_parameters,
    run_synthetic_plots,
)


In [ ]:
# Plot setup: optionally merge one test group, then compare it against CMIP.
MVO_V4P_CASE_ID = "v20260212"
MVO_V4P_RUN_PLOTS = True
MVO_V4P_TITLE_LABEL = "E3SM versus CMIP"

# Set MVO_V4P_RUN_MERGE=True when raw run JSONs should be merged before plotting.
MVO_V4P_RUN_MERGE = False
MVO_V4P_CLEAN_MERGED_OUTPUT = False
MVO_V4P_DRY_RUN_MERGE = False
MVO_V4P_DEBUG = True

# Merged metrics location and local config.
MVO_V4P_METRICS_DATA_ROOT = Path("/global/cfs/cdirs/e3sm/diagnostics/pcmdi_data/metrics_data")
MVO_V4P_METRIC_CONFIG_FILE = PROJECT_ROOT / "config" / "synthetic_metrics_list.json"
MVO_V4P_OUTPUT_DIR = Path("/global/cfs/cdirs/e3sm/www/zhan391/eamxx-pcmdi/analysis_output")

# Raw PCMDI diagnostic runs to merge. Each entry matches the PCMDI run setup:
# raw metrics are read from www / case / "pcmdi_diags" / run_type / "metrics_data".
MVO_V4P_TEST_COMBINED = False
MVO_V4P_TEST_MODEL_ONLY = False
MVO_V4P_TEST_MIP = "e3sm"
MVO_V4P_TEST_EXP = "historical"
MVO_V4P_TEST_MERGED_GROUP = "climo"
MVO_V4P_TEST_GROUP_LABEL = "E3SM"
MVO_V4P_TEST_RUNS = [
    PCMDIRun(
        case="20231209.v3.LR.piControl-spinup.chrysalis",
        model_name="EAMXX-ne256_coupled-test",
        metrics_case_id="v20260531",
        output_name="v3.LR.CPL",
        www=Path("/global/cfs/cdirs/e3sm/www/zhan391/eamxx-pcmdi"),
    ),
    PCMDIRun(
        case="20250906.wcycl1850.ne120pg2_r025_RRSwISC6to18E3r5.test6.1.chrysalis",
        model_name="v3-HR_test6-1",
        metrics_case_id="v20260601",
        output_name="v3.HR.CPL",
        www=Path("/global/cfs/cdirs/e3sm/www/zhan391/eamxx-pcmdi"),
    ),
    PCMDIRun(
        case="20260204.ne256.WCYCLXX1850.SOI",
        model_name="EAMXX-ne256_coupled-test",
        metrics_case_id="v20260601",
        output_name="v4P.CPL",
        www=Path("/global/cfs/cdirs/e3sm/www/zhan391/eamxx-pcmdi"),
    ),
    PCMDIRun(
        case="ne256pg2_ne256pg2.F20TR-SCREAMv1.July-1.spanc800.2xauto.acc150.n0032.test2.1",
        model_name="EAMXX_test2_1",
        metrics_case_id="v20260531",
        output_name="v4P.AMIP",
        www=Path("/global/cfs/cdirs/e3sm/www/zhan391/eamxx-pcmdi"),
    ),
]
MVO_V4P_TEST_HIGHLIGHT_MODELS = [run.output_name or run.model_name for run in MVO_V4P_TEST_RUNS]

MVO_V4P_TEST_RAW_GROUP = MetricsGroup(
    name=MVO_V4P_TEST_MERGED_GROUP,
    model_pattern="*",
    runs=MVO_V4P_TEST_RUNS,
    mips=[MVO_V4P_TEST_MIP],
    exps=[MVO_V4P_TEST_EXP],
    case_id=MVO_V4P_CASE_ID,
    model_rename=None,
    movs_years=(1850, 2014),
    movs_modes="NAM,NAO,PNA,NPO,SAM,PSA1,PSA2,PDO,NPGO,AMO",
    movs_obses=(
        "NOAA-20C,NOAA-20C,NOAA-20C,NOAA-20C,NOAA-20C,NOAA-20C,NOAA-20C,"
        "HadISST,HadISST,HadISST"
    ),
    enso_collections="ENSO_perf,ENSO_tel,ENSO_proc",
    enso_obses="ERA5,ERA5,ERA5",
)


In [ ]:
if MVO_V4P_RUN_MERGE:
    merge_metrics_group(
        MVO_V4P_TEST_RAW_GROUP,
        metrics_root=MVO_V4P_METRICS_DATA_ROOT,
        run_type="model_vs_obs",
        enable_clim=True,
        enable_movs=True,
        enable_enso=True,
        strict=True,
        verbose=True,
        dry_run=MVO_V4P_DRY_RUN_MERGE,
        clean=MVO_V4P_CLEAN_MERGED_OUTPUT,
    )
    MVO_V4P_TEST_COMBINED = True

# Merged test group consumed by the plot workflow.
MVO_V4P_TEST_DATASET = dataset_for_merged_group(
    mip=MVO_V4P_TEST_MIP,
    group=MVO_V4P_TEST_MERGED_GROUP,
    case_id=MVO_V4P_CASE_ID,
    root=MVO_V4P_METRICS_DATA_ROOT,
)

# Reference setup. Use built-in CMIP defaults from MVO_V4P_METRICS_DATA_ROOT.
MVO_V4P_REF_GROUP_LABEL = "CMIP"
MVO_V4P_REF_DATASET = None
MVO_V4P_ALIGN_WITH_CMIP = True

MVO_V4P_SHOW_MEAN_COLUMNS = False
MVO_V4P_MEAN_GROUP1_NAME = None #"CMIP (mean)"
MVO_V4P_MEAN_GROUP2_NAME = None #"E3SM (mean)"

# Diagnostics to generate.
MVO_V4P_CLIM_VIEWER = True
MVO_V4P_MOVA_VIEWER = True
MVO_V4P_MOVC_VIEWER = True
MVO_V4P_ENSO_VIEWER = True

# Plot settings.
MVO_V4P_CLIM_VARS = (
    "pr,prw,psl,rlds,rldscs,rltcre,rstcre,rsus,rsuscs,rlus,rlut,rlutcs,"
    "rsds,rsdscs,rsut,rsutcs,rtmt,sfcWind,tas,tauu,tauv,ts,ta-200,ta-850,"
    "ua-200,ua-850,va-200,va-850,zg-500"
)
MVO_V4P_CLIM_REGIONS = "global"
MVO_V4P_MOVS_GROUP = "cbf"
MVO_V4P_ATM_MODES = None
MVO_V4P_ATM_OBS = "NOAA-20C"
MVO_V4P_CPL_MODES = None
MVO_V4P_CPL_OBS = "HadISST"
MVO_V4P_ERROR_NORM = "reference"
MVO_V4P_FIGURE_FORMAT = "pdf"

MVO_V4P_EXCLUDE_VARS = {
    "E3SM-1-0": ["ta-850"],
    "E3SM-1-1-ECA": ["ta-850"],
    "CIESM": ["pr"],
    "KIOST-ESM": ["zg-500", "ta-850"],
    "GISS-E2-2-G": ["rlutcs", "zg-500"],
}
MVO_V4P_EXCLUDE_MODELS = ["E3SM-1-0", "E3SM-1-1", "E3SM-1-1-ECA", "E3SM-2-0", "E3SM-2-1"]
MVO_V4P_EXTRA_GROUPS_NAME = [""]

mvo_v4p_parameters = make_comparison_parameters(
    case_id=MVO_V4P_CASE_ID,
    ref_group=MVO_V4P_REF_GROUP_LABEL,
    test_group=MVO_V4P_TEST_GROUP_LABEL,
    ref_dataset=MVO_V4P_REF_DATASET,
    test_dataset=MVO_V4P_TEST_DATASET,
    test_highlight_models=MVO_V4P_TEST_HIGHLIGHT_MODELS,
    metrics_root=MVO_V4P_METRICS_DATA_ROOT,
    out_dir=MVO_V4P_OUTPUT_DIR,
    align_with_cmip=MVO_V4P_ALIGN_WITH_CMIP,
    test_combined=MVO_V4P_TEST_COMBINED,
    test_model_only=MVO_V4P_TEST_MODEL_ONLY,
    show_mean_columns=MVO_V4P_SHOW_MEAN_COLUMNS,
    clim_vars=MVO_V4P_CLIM_VARS,
    clim_regions=MVO_V4P_CLIM_REGIONS,
    movs_group=MVO_V4P_MOVS_GROUP,
    error_norm=MVO_V4P_ERROR_NORM,
    exclude_vars=MVO_V4P_EXCLUDE_VARS,
    exclude_models=MVO_V4P_EXCLUDE_MODELS,
    atm_modes=MVO_V4P_ATM_MODES,
    atm_obs=MVO_V4P_ATM_OBS,
    cpl_modes=MVO_V4P_CPL_MODES,
    cpl_obs=MVO_V4P_CPL_OBS,
    figure_format=MVO_V4P_FIGURE_FORMAT,
    clim_viewer=MVO_V4P_CLIM_VIEWER,
    mova_viewer=MVO_V4P_MOVA_VIEWER,
    movc_viewer=MVO_V4P_MOVC_VIEWER,
    enso_viewer=MVO_V4P_ENSO_VIEWER,
    mean_group1_name=MVO_V4P_MEAN_GROUP1_NAME,
    mean_group2_name=MVO_V4P_MEAN_GROUP2_NAME,
    extra_groups_name=MVO_V4P_EXTRA_GROUPS_NAME,
)

if MVO_V4P_RUN_PLOTS:
    run_synthetic_plots(
        mvo_v4p_parameters,
        title_label=MVO_V4P_TITLE_LABEL,
        metric_file=MVO_V4P_METRIC_CONFIG_FILE,
        debug=MVO_V4P_DEBUG,
    )
else:
    print("Set MVO_V4P_RUN_PLOTS = True to generate figures.")
